# Business Recommendations: Retention and Customer Value Strategy

In this final notebook, we combine the outputs from the earlier customer analytics workflows to generate actionable business recommendations.

So far, the project has produced:

- a customer base table,
- RFM-based customer segments,
- churn risk predictions,
- customer lifetime value (CLV) estimates.

The purpose of this notebook is to translate those analytical outputs into a practical customer strategy.

We focus on answering business questions such as:

- Which customers should be prioritized for retention?
- Which customers are valuable but at risk of churn?
- Where should the business spend retention budget?
- What actions make sense for different customer groups?

This notebook therefore moves from **predictive analytics** to **decision-oriented customer strategy**.

We begin by importing the libraries needed for data manipulation, analysis, and visualization.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

We now load the main outputs generated in earlier notebooks:

- `customer_base.csv` from the customer feature engineering step,
- `customer_churn_scores.csv` from the churn prediction notebook,
- `customer_clv_scores.csv` from the CLV modeling notebook.

These files will be combined into a single customer strategy table.

In [ ]:
DATA_DIR = Path("../data/processed")

customer_base = pd.read_csv(DATA_DIR / "customer_base.csv")
customer_segments = pd.read_csv(DATA_DIR / "customer_segments.csv")
churn_scores = pd.read_csv(DATA_DIR / "customer_churn_scores.csv")
clv_scores = pd.read_csv(DATA_DIR / "customer_clv_scores.csv")

customer_base.shape, customer_segments.shape, churn_scores.shape, clv_scores.shape

The RFM segmentation was created in Notebook 03, so we merge it here.

In [ ]:
customer_base = customer_base.merge(
    customer_segments[["customer_unique_id", "segment"]],
    on="customer_unique_id",
    how="left"
)

customer_base.head()

Before merging the datasets, we inspect their columns to confirm the join keys and available business features.

In [ ]:
print("customer_base columns:")
print(customer_base.columns.tolist())

print("\nchurn_scores columns:")
print(churn_scores.columns.tolist())

print("\nclv_scores columns:")
print(clv_scores.columns.tolist())

The CLV scores file was saved with the customer identifier as the index, so when reloaded it may appear as an unnamed column.

We standardize the CLV table so that `customer_unique_id` is available as a normal column for merging.

In [ ]:
if "customer_unique_id" not in clv_scores.columns:
    first_col = clv_scores.columns[0]
    clv_scores = clv_scores.rename(columns={first_col: "customer_unique_id"})

clv_scores.head()

We select the most relevant variables from each source table.

From the customer base table, we keep descriptive customer behavior fields such as:
- recency,
- order frequency,
- total spend,
- RFM segment.

From the churn and CLV tables, we keep:
- churn probability,
- churn risk band,
- predicted CLV,
- CLV segment,
- value matrix labels if already created.

In [ ]:
base_cols = [
    "customer_unique_id",
    "recency_days",
    "n_orders",
    "total_spend",
    "avg_order_value",
    "segment"
]

churn_cols = [
    "customer_unique_id",
    "churn_probability",
    "churn_risk_band"
]

clv_cols = [
    "customer_unique_id",
    "frequency",
    "recency",
    "T",
    "monetary_value",
    "predicted_purchases_6m",
    "clv_6m",
    "clv_segment"
]

extra_clv_cols = [col for col in ["clv_tier", "churn_risk", "value_segment"] if col in clv_scores.columns]
clv_cols = clv_cols + extra_clv_cols

customer_strategy = (
    customer_base[base_cols]
    .merge(churn_scores[churn_cols], on="customer_unique_id", how="left")
    .merge(clv_scores[clv_cols], on="customer_unique_id", how="left")
)

customer_strategy.shape

We inspect the merged customer strategy table to verify that the key customer analytics outputs are now available in one place.

In [ ]:
customer_strategy.head()

After merging, we inspect missing values.

Some customers may have missing CLV or churn information due to modeling constraints, such as insufficient repeat purchase history or differing snapshot coverage.

In [ ]:
customer_strategy.isna().mean().sort_values(ascending=False)

To make the business recommendation layer usable for all customers, we fill missing categorical labels with defaults and keep numerical missing values as needed for later rules.

For example:
- customers without CLV segments can be treated as lower-information customers,
- customers without value matrix labels can be reassigned using fresh rules in this notebook.

In [ ]:
customer_strategy["clv_segment"] = customer_strategy["clv_segment"].fillna("Low Value")
customer_strategy["churn_risk_band"] = customer_strategy["churn_risk_band"].fillna("Low")

To ensure consistency, we now create a simple customer value matrix based on:

- predicted churn risk
- predicted customer lifetime value

This matrix is one of the most useful frameworks in retention strategy because it separates customers into groups that need different actions.

In [ ]:
customer_strategy["clv_tier_strategy"] = np.where(
    customer_strategy["clv_6m"] >= customer_strategy["clv_6m"].median(),
    "High CLV",
    "Low CLV"
)

customer_strategy["churn_risk_strategy"] = np.where(
    customer_strategy["churn_probability"] >= 0.60,
    "High Risk",
    "Low Risk"
)

customer_strategy["value_segment_strategy"] = (
    customer_strategy["clv_tier_strategy"]
    + " | "
    + customer_strategy["churn_risk_strategy"]
)

customer_strategy["value_segment_strategy"].value_counts()

A quick count of customers by value segment helps show the overall strategic landscape of the customer base.

In [ ]:
customer_strategy["value_segment_strategy"].value_counts().plot(kind="bar")
plt.title("Customer Value Matrix")
plt.xlabel("Value Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=20)
plt.show()

Counts alone are not enough. We also want to know how much historical spend is represented by each value segment.

This helps distinguish small but valuable groups from large low-value groups.

In [ ]:
value_matrix_summary = (
    customer_strategy.groupby("value_segment_strategy")
    .agg(
        customers=("customer_unique_id", "nunique"),
        avg_total_spend=("total_spend", "mean"),
        avg_clv_6m=("clv_6m", "mean"),
        avg_churn_probability=("churn_probability", "mean")
    )
    .sort_values("avg_clv_6m", ascending=False)
)

value_matrix_summary

Businesses rarely target all customers equally.

Instead, retention campaigns should prioritize customers based on:
- expected future value,
- risk of churn,
- expected effectiveness of intervention,
- campaign cost.

We therefore estimate a simple retention ROI framework using churn probability and predicted CLV.

In [ ]:
RETENTION_COST = 15.0
RETENTION_SUCCESS_RATE = 0.20

customer_strategy["expected_retained_value"] = (
    customer_strategy["clv_6m"]
    * customer_strategy["churn_probability"]
    * RETENTION_SUCCESS_RATE
)

customer_strategy["expected_roi"] = (
    customer_strategy["expected_retained_value"] - RETENTION_COST
)

customer_strategy[[
    "customer_unique_id",
    "clv_6m",
    "churn_probability",
    "expected_retained_value",
    "expected_roi"
]].head()

We now rank customers by expected retention ROI.

This gives a direct prioritization list for retention targeting, where higher values indicate greater expected return after accounting for campaign cost.

In [ ]:
top_retention_targets = (
    customer_strategy.sort_values("expected_roi", ascending=False)
    .reset_index(drop=True)
)

top_retention_targets.head(10)

To make the analysis more actionable, we estimate the expected value of targeting the top customer group.

This provides a simple example of how model outputs can be translated into budget and campaign planning.

In [ ]:
top_n = 1000

campaign_summary = top_retention_targets.head(top_n).agg({
    "expected_retained_value": "sum",
    "expected_roi": "sum"
}).to_frame(name="value")

campaign_summary.loc["target_customers", "value"] = top_n
campaign_summary

Next, we translate model outputs into concrete customer actions.

We define four business action groups:

- **Retain Immediately**: high value and high churn risk
- **Loyalty / Upsell**: high value and low churn risk
- **Low-Cost Retention**: low value and high churn risk
- **Automated Nurture**: low value and low churn risk

This makes the output easy for business stakeholders to interpret and use.

In [ ]:
conditions = [
    (customer_strategy["clv_tier_strategy"] == "High CLV") &
    (customer_strategy["churn_risk_strategy"] == "High Risk"),

    (customer_strategy["clv_tier_strategy"] == "High CLV") &
    (customer_strategy["churn_risk_strategy"] == "Low Risk"),

    (customer_strategy["clv_tier_strategy"] == "Low CLV") &
    (customer_strategy["churn_risk_strategy"] == "High Risk"),

    (customer_strategy["clv_tier_strategy"] == "Low CLV") &
    (customer_strategy["churn_risk_strategy"] == "Low Risk")
]

choices = [
    "Retain Immediately",
    "Loyalty / Upsell",
    "Low-Cost Retention",
    "Automated Nurture"
]

customer_strategy["recommended_action"] = np.select(
    conditions,
    choices,
    default="Monitor"
)

customer_strategy["recommended_action"].value_counts()

We summarize the business action groups to understand their size and economic importance.

This helps explain where the business should focus attention and where low-touch automation may be enough.

In [ ]:
action_summary = (
    customer_strategy.groupby("recommended_action")
    .agg(
        customers=("customer_unique_id", "nunique"),
        avg_churn_probability=("churn_probability", "mean"),
        avg_clv_6m=("clv_6m", "mean"),
        avg_total_spend=("total_spend", "mean"),
        total_expected_roi=("expected_roi", "sum")
    )
    .sort_values("total_expected_roi", ascending=False)
)

action_summary

This chart shows how customers are distributed across the recommended action groups.

In [ ]:
customer_strategy["recommended_action"].value_counts().plot(kind="bar")
plt.title("Recommended Customer Actions")
plt.xlabel("Action Group")
plt.ylabel("Number of Customers")
plt.xticks(rotation=20)
plt.show()

The most strategically important customers are typically those with both:

- high predicted lifetime value
- high churn risk

We inspect this segment directly because it represents the most urgent retention opportunity.

In [ ]:
high_value_high_risk = customer_strategy.loc[
    customer_strategy["recommended_action"] == "Retain Immediately"
].sort_values("expected_roi", ascending=False)

high_value_high_risk.head(10)

To support communication with non-technical stakeholders, we build a simple recommendation table linking each customer group to a suggested action.

In [ ]:
recommendation_table = pd.DataFrame({
    "segment": [
        "Retain Immediately",
        "Loyalty / Upsell",
        "Low-Cost Retention",
        "Automated Nurture"
    ],
    "business_meaning": [
        "High-value customers with elevated churn risk",
        "High-value customers with relatively low churn risk",
        "Lower-value customers at risk of churn",
        "Lower-value customers with low churn risk"
    ],
    "recommended_strategy": [
        "Provide proactive retention offers, targeted outreach, or priority support",
        "Use loyalty rewards, personalized recommendations, and cross-sell campaigns",
        "Use lightweight incentives such as automated coupons or reminders",
        "Use low-cost lifecycle messaging and automated engagement campaigns"
    ]
})

recommendation_table

We save the final customer strategy table and summary outputs so they can be reused in dashboards, presentations, or downstream analysis.

In [ ]:
customer_strategy.to_csv(DATA_DIR / "customer_strategy_table.csv", index=False)
value_matrix_summary.to_csv(DATA_DIR / "value_matrix_summary.csv")
action_summary.to_csv(DATA_DIR / "action_summary.csv")
recommendation_table.to_csv(DATA_DIR / "recommendation_table.csv", index=False)

## Final Recommendations

Based on the combined RFM, churn, and CLV analysis, the following business recommendations emerge:

### 1. Prioritize high-value customers at high risk of churn
These customers represent the most important retention group because they combine strong expected future value with elevated churn likelihood.

### 2. Protect loyal high-value customers with loyalty and upsell strategies
Customers with high expected value but lower churn risk should not be ignored. They are strong candidates for loyalty programs, exclusive offers, and personalized recommendations.

### 3. Use low-cost retention tactics for lower-value at-risk customers
For customers with lower predicted CLV, retention should remain selective and cost-conscious. Automated offers or reminder campaigns are more suitable than expensive intervention.

### 4. Maintain lightweight nurture programs for low-risk, low-value customers
These customers do not require major retention spend, but they should remain engaged through scalable lifecycle marketing.

### 5. Use churn and CLV together for decision-making
Churn probability alone is not enough. The most effective customer strategy comes from combining churn risk with customer value, which allows retention resources to be allocated more efficiently.

Overall, this notebook demonstrates how customer analytics can move beyond descriptive reporting and predictive modeling into concrete business action.